In [3]:
import numpy as np
import pandas as pd


In [4]:
df = pd.read_csv("CarPrice_Assignment.csv")
df = df.sample(frac=1).reset_index(drop=True)

print(df.head())
print("Dataset shape:", df.shape)


   car_ID  symboling      CarName fueltype aspiration doornumber    carbody  \
0     150          0    subaru dl      gas      turbo       four      wagon   
1      95          1  nissan leaf      gas        std        two      sedan   
2       6          2     audi fox      gas        std        two      sedan   
3      96          1  nissan juke      gas        std        two  hatchback   
4      50          0    jaguar xk      gas        std        two      sedan   

  drivewheel enginelocation  wheelbase  ...  enginesize  fuelsystem  \
0        4wd          front       96.9  ...         108        mpfi   
1        fwd          front       94.5  ...          97        2bbl   
2        fwd          front       99.8  ...         136        mpfi   
3        fwd          front       94.5  ...          97        2bbl   
4        rwd          front      102.0  ...         326        mpfi   

   boreratio  stroke compressionratio horsepower  peakrpm citympg  highwaympg  \
0       3.62    2

In [5]:
category_columns = [
    'fueltype',
    'aspiration',
    'doornumber',
    'carbody',
    'drivewheel',
    'enginelocation',
    'enginetype',
    'cylindernumber',
    'fuelsystem'
]

df_encoded = pd.get_dummies(
    df,
    columns=category_columns,
    dtype=int
)

x = df_encoded.drop(
    columns=['price', 'car_ID', 'CarName']
).values.astype(float)

y = df_encoded['price'].values.astype(float)

print("Input shape:", x.shape)
print("Target shape:", y.shape)

Input shape: (205, 52)
Target shape: (205,)


In [6]:
p = len(x)
split = int(0.8 * p)

x_train = x[:split]
y_train = y[:split]

x_test = x[split:]
y_test = y[split:]

print("Training data:", x_train.shape)
print("Testing data:", x_test.shape)

Training data: (164, 52)
Testing data: (41, 52)


In [7]:
x_mean = np.mean(x_train, axis=0)
x_std = np.std(x_train, axis=0)

x_std[x_std == 0] = 1

x_train = (x_train - x_mean) / x_std
x_test = (x_test - x_mean) / x_std

y_mean = np.mean(y_train)
y_std = np.std(y_train)

y_train = (y_train - y_mean) / y_std
y_test_original = y_test.copy()
y_test_scaled = (y_test - y_mean) / y_std



In [8]:
l = x_train.shape[1] 

w = np.zeros(l)
b = 0.0

print("Number of features:", l)

Number of features: 52


In [9]:
def compute_cost(x, y, w, b):
    m = x.shape[0]
    predictions = np.dot(x, w) + b
    errors = predictions - y
    total_cost = np.sum(errors ** 2) / (2 * m)
    return total_cost

In [10]:
def gradient_descent(x, y, w, b, alpha, iterations):
    m = x.shape[0]

    for _ in range(iterations):
        predictions = np.dot(x, w) + b
        errors = predictions - y

        dj_dw = np.dot(x.T, errors) / m
        dj_db = np.mean(errors)

        w = w - alpha * dj_dw
        b = b - alpha * dj_db

    return w, b

In [11]:
alphas = [0.001, 0.003, 0.01, 0.03]
iteration_values = [500, 1500, 3000]

results_tuning = []

best_rmse = float("inf")
best_w = None
best_b = None

for alpha_value in alphas:
    for iteration_value in iteration_values:

        w_temp = np.zeros(x_train.shape[1])
        b_temp = 0.0

        w_temp, b_temp = gradient_descent(
            x_train,
            y_train,
            w_temp,
            b_temp,
            alpha_value,
            iteration_value
        )

        test_predictions_scaled = np.dot(x_test, w_temp) + b_temp

        test_predictions_temp = (
            test_predictions_scaled * y_std + y_mean
        )

        test_rmse_temp = np.sqrt(
            np.mean(
                (y_test_original - test_predictions_temp) ** 2
            )
        )

        results_tuning.append([
            alpha_value,
            iteration_value,
            test_rmse_temp
        ])

        if test_rmse_temp < best_rmse:
            best_rmse = test_rmse_temp
            best_alpha = alpha_value
            best_iterations = iteration_value
            best_w = w_temp.copy()
            best_b = b_temp

tuning_df = pd.DataFrame(
    results_tuning,
    columns=['Alpha', 'Iterations', 'Test RMSE']
)

tuning_df = tuning_df.sort_values(
    by='Test RMSE'
).reset_index(drop=True)

print(tuning_df.to_string(index=False))

print("\nBest Hyperparameters")
print("Alpha:", best_alpha)
print("Iterations:", best_iterations)
print("Test RMSE:", best_rmse)

 Alpha  Iterations   Test RMSE
 0.030        1500 3156.150252
 0.010        3000 3157.023150
 0.030         500 3257.002512
 0.030        3000 3257.150357
 0.010        1500 3257.221853
 0.003        3000 3390.500421
 0.010         500 3568.619618
 0.003        1500 3599.995645
 0.001        3000 3710.150668
 0.001        1500 3848.964759
 0.003         500 3848.978555
 0.001         500 4034.077335

Best Hyperparameters
Alpha: 0.03
Iterations: 1500
Test RMSE: 3156.1502519698874


In [12]:
w = best_w.copy()
b = best_b

cost = compute_cost(x_train, y_train, w, b)

print("Final Training Cost:", cost)

test_predictions_scaled = np.dot(x_test, w) + b

test_predictions = (
    test_predictions_scaled * y_std + y_mean
)

test_rmse = np.sqrt(
    np.mean((y_test_original - test_predictions) ** 2)
)

test_mae = np.mean(
    np.abs(y_test_original - test_predictions)
)

ss_res = np.sum(
    (y_test_original - test_predictions) ** 2
)

ss_tot = np.sum(
    (y_test_original - np.mean(y_test_original)) ** 2
)

test_r2 = (
    1 - ss_res / ss_tot
    if ss_tot != 0
    else float("nan")
)

print("\nFinal Model Results")
print("Best Alpha:", best_alpha)
print("Best Iterations:", best_iterations)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R²:", test_r2)

Final Training Cost: 0.02193782723903491

Final Model Results
Best Alpha: 0.03
Best Iterations: 1500
Test RMSE: 3156.1502519698874
Test MAE: 2195.2239510672225
Test R²: 0.8119019642378741


In [13]:
comparison_df = pd.DataFrame({
    'Actual Price': y_test_original,
    'Predicted Price': test_predictions,
    'Error': y_test_original - test_predictions
})

print(comparison_df.head(10).to_string(index=False))

 Actual Price  Predicted Price        Error
      16925.0     13202.244338  3722.755662
      14869.0     13971.288688   897.711312
       8921.0     10057.439395 -1136.439395
       6295.0      6017.570421   277.429579
      16845.0     18665.219883 -1820.219883
      12940.0     17129.555455 -4189.555455
      17710.0     20720.607524 -3010.607524
      15510.0     14142.280086  1367.719914
      16630.0     12606.609631  4023.390369
      17199.0     16716.863628   482.136372
